# 02 — TF‑IDF + Sentiment multilingue → Modèle ML (marchés)

Ce notebook construit un indicateur à partir d'un CSV de textes (`text`) et d'un `prices.csv`, puis entraîne un modèle pour prédire le mouvement **J+1** (classification) ou le **retour J+1** (régression).

In [1]:
# Install packages
%pip install -q pandas numpy scikit-learn transformers torch tqdm langid joblib langdetect

import os, json, sqlite3, hashlib
import numpy as np
import pandas as pd
from tqdm import tqdm

import langid
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, roc_auc_score, r2_score, mean_squared_error
from datetime import timedelta


from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline


Note: you may need to restart the kernel to use updated packages.


In [2]:
# --- General Parameters  ---
TEXTS_PATH = "all_ECB_conferences.csv"   #contains date,title,link,text
PRICES_PATH = "prices.csv" #contains timestamp,asset,price
BUCKET = "D"               # 'D' (day)
HORIZON = 1                 
TASK = "classification"     # 'classification' or 'regression'

MDL_EN = "ProsusAI/finbert" #Finbert for text in English
MDL_ML = "cardiffnlp/twitter-xlm-roberta-base-sentiment" #XLM-RoBERTa for multi-lingual text
# ECB texts are in different EU languages

TFIDF_MIN_DF = 2
TFIDF_MAX_DF = 0.98
NGRAMS = (3,5)
SVD_COMP = 150
BATCH = 64


In [3]:

def read_texts(path, col_text="text", col_ts="timestamp", col_asset="asset", bucket=BUCKET):
    df = pd.read_csv(path, sep=",", encoding="utf-8", on_bad_lines="skip")
    assert col_text in df.columns, f"Colonne '{col_text}' absente de {path}"
    if col_ts not in df.columns:
        df[col_ts] = pd.Timestamp.utcnow().normalize()
    if col_asset not in df.columns:
        df[col_asset] = "STOCK"
    df[col_ts] = pd.to_datetime(df[col_ts], errors="coerce")
    df["bucket"] = df[col_ts].dt.to_period(bucket).dt.to_timestamp()
    return df[[col_asset,"bucket",col_text]].rename(columns={col_asset:"asset", col_text:"text"})


def read_prices(path, col_ts="timestamp", col_asset="asset", col_px="price", bucket=BUCKET):
    df = pd.read_csv(path, sep=",", encoding="utf-8", on_bad_lines="skip")
    for c in (col_ts,col_asset,col_px):
        assert c in df.columns, f"Colonne '{c}' absente de {path}"
    df[col_ts] = pd.to_datetime(df[col_ts], errors="coerce")
    df["bucket"] = df[col_ts].dt.to_period(bucket).dt.to_timestamp()
    px = (df.sort_values([col_asset,"bucket",col_ts])
            .groupby([col_asset,"bucket"], as_index=False).tail(1)[[col_asset,"bucket",col_px]])
    return px.rename(columns={col_asset:"asset", col_px:"price"})


df_texts = read_texts(TEXTS_PATH)
df_prices = read_prices(PRICES_PATH)
print(df_texts.head())
print(df_prices.head())

   asset     bucket                                               text
0  STOCK 2025-11-05  Willem F. Duisenberg, President of the Europea...
1  STOCK 2025-11-05  Willem F. Duisenberg, President of the Europea...
2  STOCK 2025-11-05  Willem F. Duisenberg, President of the Europea...
3  STOCK 2025-11-05  Willem F. Duisenberg, President of the Europea...
4  STOCK 2025-11-05  Willem F. Duisenberg, President of the Europea...
    asset     bucket     price
0  EURUSD 2003-12-01  1.196501
1  EURUSD 2003-12-02  1.208897
2  EURUSD 2003-12-03  1.212298
3  EURUSD 2003-12-04  1.208094
4  EURUSD 2003-12-05  1.218695


C:\Users\Garance Latieule\AppData\Local\Temp\ipykernel_93228\802126127.py:9: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["bucket"] = df[col_ts].dt.to_period(bucket).dt.to_timestamp()


In [4]:

try:
    torch.set_num_threads(max(1, os.cpu_count() // 2))
except Exception:
    pass

BATCH = 16
CSV_PATH = "all_ECB_conferences.csv"  # <-- mets ton chemin
df_raw = pd.read_csv(CSV_PATH)
df_raw["text"] = df_raw["text"].fillna("").astype(str).str.replace("\r\n", "\n")


# --- Detect languages ---
# As recall : for English MDL_EN = "ProsusAI/finbert" and for other languages "cardiffnlp/twitter-xlm-roberta-base-sentiment"

try:
    from langdetect import detect
except ImportError:
    raise ImportError("Install 'langdetect' :  pip install langdetect")

def _detect_lang(txt):
    try:
        return detect(txt)
    except Exception:
        return "unknown"

# "unknown" if no text
tqdm.pandas(desc="langdetect")
df_raw["lang"] = df_raw["text"].progress_apply(lambda t: _detect_lang(t) if t.strip() else "unknown")
# Normalize : if not in english 'en'
df_raw["lang"] = df_raw["lang"].apply(lambda x: "en" if x=="en" else "other")


# --- MODEL for sentiment ---
DEV = 0 if torch.cuda.is_available() else -1  

tok_en = AutoTokenizer.from_pretrained(MDL_EN)
mod_en = AutoModelForSequenceClassification.from_pretrained(MDL_EN)
pipe_en = TextClassificationPipeline(model=mod_en, tokenizer=tok_en, device=DEV)

tok_ml = AutoTokenizer.from_pretrained(MDL_ML)
mod_ml = AutoModelForSequenceClassification.from_pretrained(MDL_ML)
pipe_ml = TextClassificationPipeline(model=mod_ml, tokenizer=tok_ml, device=DEV)

# Finbert : Max length 512
MAX_LEN_EN = getattr(tok_en, "model_max_length", 512)
if MAX_LEN_EN is None or MAX_LEN_EN > 100000:
    MAX_LEN_EN = 512

# XLM-R / RoBERTa : Max length 514
MAX_LEN_ML = getattr(tok_ml, "model_max_length", 514)
if MAX_LEN_ML is None or MAX_LEN_ML > 100000:
    MAX_LEN_ML = 514


# Cache SQLite
CACHE_DB = "sent_cache.sqlite"
con = sqlite3.connect(CACHE_DB)
con.execute("CREATE TABLE IF NOT EXISTS cache (k TEXT PRIMARY KEY, v TEXT NOT NULL)")
con.commit()

def _key(model_name, txt):
    return hashlib.sha1((model_name+"||"+txt).encode("utf-8")).hexdigest()

def _scores_from_predlist(pred_list):
    d = {x["label"].lower(): float(x["score"]) for x in pred_list}
    if "label_0" in d or "label 0" in d:
        neg = d.get("label_0", d.get("label 0", 0.0))
        neu = d.get("label_1", d.get("label 1", 0.0))
        pos = d.get("label_2", d.get("label 2", 0.0))
    else:
        pos = d.get("positive", d.get("pos", 0.0))
        neu = d.get("neutral",  d.get("neu", 0.0))
        neg = d.get("negative", d.get("neg", 0.0))
    return {"pos": pos, "neu": neu, "neg": neg, "polarity": pos - neg}

def _cache_get(keys):
    if not keys:
        return {}
    q = ",".join(["?"]*len(keys))
    cur = con.execute(f"SELECT k,v FROM cache WHERE k IN ({q})", keys)
    return dict(cur.fetchall())

def _cache_set(mapping):
    if not mapping:
        return
    con.executemany("INSERT OR REPLACE INTO cache(k,v) VALUES (?,?)",
                    list(mapping.items()))
    con.commit()


# --- Scoring multi-langues ---
def score_multilang_from_df(df_texts, batch=BATCH):
    texts = df_texts["text"].fillna("").astype(str).tolist()
    langs = df_texts["lang"].tolist()
    out = [None]*len(texts)

    keys_en = [_key(MDL_EN, t) for t,l in zip(texts, langs) if l == "en"]
    keys_ml = [_key(MDL_ML, t) for t,l in zip(texts, langs) if l != "en"]
    cached = _cache_get(keys_en + keys_ml)
    new_entries = {}

    for i,(t,l) in enumerate(zip(texts, langs)):
        k = _key(MDL_EN, t) if l == "en" else _key(MDL_ML, t)
        if k in cached:
            out[i] = json.loads(cached[k])

    def _process_idxs(idxs, pipe, model_name, max_len):
        for start in tqdm(range(0, len(idxs), batch), desc=model_name):
            sub = idxs[start:start+batch]
            preds = pipe(
                [texts[j] for j in sub],
                top_k=None,             # remplace return_all_scores
                truncation=True,
                padding=True,
                max_length=max_len,
                batch_size=min(batch, len(sub)),
                return_token_type_ids=False  
            )
            for j,p in zip(sub, preds):
                sc = _scores_from_predlist(p)
                out[j] = sc
                new_entries[_key(model_name, texts[j])] = json.dumps(sc)

    idx_en = [i for i,(t,l) in enumerate(zip(texts, langs))
              if l == "en" and _key(MDL_EN, t) not in cached]
    idx_ml = [i for i,(t,l) in enumerate(zip(texts, langs))
              if l != "en" and _key(MDL_ML, t) not in cached]

    if idx_en: _process_idxs(idx_en, pipe_en, MDL_EN, MAX_LEN_EN)
    if idx_ml: _process_idxs(idx_ml, pipe_ml, MDL_ML, MAX_LEN_ML)

    _cache_set(new_entries)
    return pd.DataFrame(out)


# --- Apply method to ECB files ---
sent_df = score_multilang_from_df(df_raw, batch=BATCH)


df_scored = pd.concat([df_raw.reset_index(drop=True),
                       sent_df.reset_index(drop=True)], axis=1)
df_scored.head()


langdetect: 100%|██████████| 326/326 [00:09<00:00, 32.79it/s]
Device set to use cpu
Device set to use cpu


,date,title,link,text,lang,pos,neu,neg,polarity
0,1998-06-09,Willem F. Duisenberg: ECB Press conference: In...,https://www.ecb.europa.eu/press/press_conferen...,"Willem F. Duisenberg, President of the Europea...",en,0.032779,0.942236,0.024984,0.007795
1,1998-07-08,Willem F. Duisenberg: ECB Press conference: In...,https://www.ecb.europa.eu/press/press_conferen...,"Willem F. Duisenberg, President of the Europea...",en,0.733954,0.170740,0.095306,0.638648
2,1998-09-11,Willem F. Duisenberg: ECB Press conference: In...,https://www.ecb.europa.eu/press/press_conferen...,"Willem F. Duisenberg, President of the Europea...",en,0.050362,0.728460,0.221179,-0.170817
3,1998-10-13,Willem F. Duisenberg: Introductory statement w...,https://www.ecb.europa.eu/press/press_conferen...,"Willem F. Duisenberg, President of the Europea...",en,0.139786,0.747997,0.112217,0.027569
4,1998-11-03,Willem F. Duisenberg: Introductory statement w...,https://www.ecb.europa.eu/press/press_conferen...,"Willem F. Duisenberg, President of the Europea...",en,0.028792,0.062427,0.908781,-0.879990


In [5]:
# ==============================
# PnL: long si le modèle prévoit up, short sinon
# ==============================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TC_BP     = 0.0   # coût de transaction par changement de position (en bp)
ANNUALIZE = 252   # jours ouvrés; ajuste si event-days uniquement
DV01      = None  # €/bp pour convertir le PnL en euros (ex: 850)

idx_test = X_test.index

if TASK == "classification":
    # y_test : direction vraie (0/1 ou -1/+1)
    proba  = pd.Series(final_model.predict_proba(X_test)[:, 1], index=idx_test)
    # STRATÉGIE "toujours investi" :
    # long si proba >= 0.5, short sinon
    signal = (proba >= 0.5).astype(int).replace({0:-1, 1:1})

    # convertir y_test en {-1,+1} si besoin
    y_dir = pd.Series(y_test, index=idx_test)
    if sorted(pd.unique(y_dir)) == [0,1]:
        y_dir = y_dir.replace({0:-1, 1:1})
    else:
        y_dir = y_dir.astype(int)

    # décalage anti-look-ahead: position de t appliquée au move de t+1
    signal_shifted = signal.shift(1).fillna(0)

    # coûts de transaction quand la position change
    trades = (signal_shifted.diff().abs() > 0).astype(int).fillna(0)
    tc = trades * TC_BP

    # PnL en bp = +1 si bon sens, -1 si mauvais (moins coûts)
    pnl_bp = signal_shifted * y_dir - tc

else:
    # RÉGRESSION: y_test est un move (ex Δbp). Long si prédiction > 0, short sinon
    pred = pd.Series(final_model.predict(X_test), index=idx_test)
    signal = np.sign(pred).astype(int).replace(0, 1)  # 0 => neutre; si tu veux toujours investi, force 0 -> +1

    signal_shifted = signal.shift(1).fillna(0)

    trades = (signal_shifted.diff().abs() > 0).astype(int).fillna(0)
    tc = trades * TC_BP

    move_bp = pd.Series(y_test, index=idx_test).astype(float)
    pnl_bp = signal_shifted * move_bp - tc

# === Metrics & plots ===
eq = pnl_bp.cumsum()
mean_daily = pnl_bp.mean()
std_daily  = pnl_bp.std(ddof=1)
sharpe     = np.nan if std_daily == 0 else (mean_daily / std_daily) * np.sqrt(ANNUALIZE)

print("\n[Strategy PnL]")
print(f"Trades: {int(trades.sum())}")
print(f"Mean (bp/day): {mean_daily:.4f} | Std: {std_daily:.4f} | Sharpe~: {sharpe:.2f}")

if DV01 is not None:
    pnl_eur = pnl_bp * float(DV01)
    print(f"Total PnL (EUR): {pnl_eur.sum():,.0f} | Mean/day (EUR): {pnl_eur.mean():.2f}")

plt.figure(figsize=(9,4))
plt.plot(eq.index, eq.values)
plt.title("Equity curve (bp)")
plt.tight_layout(); plt.show()


NameError: name 'X_test' is not defined

In [ ]:
import pandas as pd

# ===== 1) Sentiment mensuel global (à partir de df_scored) =====
df = df_scored.copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])
df["bucket"] = df["date"].dt.to_period("D").dt.to_timestamp()

sent_m = (df.groupby("bucket", as_index=False)
            .agg(pos_mean=("pos","mean"),
                 neu_mean=("neu","mean"),
                 neg_mean=("neg","mean"),
                 polarity_mean=("polarity","mean"),
                 n_docs=("text","size")))

# ===== 2) Prix mensuels par asset + retours =====
px = pd.read_csv("prices.csv")  # colonnes: timestamp,asset,price
px["timestamp"] = pd.to_datetime(px["timestamp"])
px["bucket"] = px["timestamp"].dt.to_period("M").dt.to_timestamp()

px_m = (px.sort_values(["asset","timestamp"])
          .groupby(["asset","bucket"], as_index=False)
          .agg(price_last=("price","last")))

# Rendement 1 mois par asset
px_m["ret_1m"] = px_m.groupby("asset")["price_last"].pct_change()

# ===== 3) Merge: on associe le même sentiment global à tous les assets sur le même mois =====
df_merged = (px_m.merge(sent_m, on="bucket", how="left")
                 .sort_values(["asset","bucket"])
                 .reset_index(drop=True))

# (optionnel) décaler le sentiment d'1 mois pour tester un effet retardé
df_merged["polarity_mean_lag1"] = df_merged.groupby("asset")["polarity_mean"].shift(1)

# ===== 4) Quick check: aperçu + corrélations simples =====
print("\nCorrélations (ret_1m vs sentiment courant/retardé):")
print(df_merged[["ret_1m","polarity_mean","polarity_mean_lag1"]].corr())



Corrélations (ret_1m vs sentiment courant/retardé):
                      ret_1m  polarity_mean  polarity_mean_lag1
ret_1m              1.000000      -0.137671           -0.564376
polarity_mean      -0.137671       1.000000                 NaN
polarity_mean_lag1 -0.564376            NaN            1.000000


In [ ]:
# --- 0) on repart des deux DF : df_scored (discours scorés) et df_prices (spot journalier) ---
# df_prices a au moins ['asset','bucket','price'] (ou 'timestamp' -> on fabrique bucket)

# 1) sentiment agrégé par bucket (choisis period "D" ou "M")
PERIOD = "M"  # "D" journalier, "M" mensuel
df_s = df_scored.copy()
df_s["date"] = pd.to_datetime(df_s["date"], errors="coerce")
df_s = df_s.dropna(subset=["date"])
df_s["bucket"] = df_s["date"].dt.to_period(PERIOD).dt.to_timestamp()

# a) agrégat texte (joint tous les textes du bucket)
agg_text = (
    df_s.groupby("bucket")["text"]
        .apply(lambda s: " \n ".join(s.dropna().astype(str)))
        .reset_index(name="text")              # <-- on force le nom 'text'
)

# b) agrégat sentiments
agg_sent = (
    df_s.groupby("bucket", as_index=False)
        .agg(pos_mean=("pos","mean"),
             neu_mean=("neu","mean"),
             neg_mean=("neg","mean"),
             polarity_mean=("polarity","mean"),
             n_docs=("text","size"))
)

# c) feature table (sentiments + texte) AU NIVEAU bucket
feat_text = agg_sent.merge(agg_text, on="bucket", how="left")

# 2) prix -> s’assurer d’avoir 'bucket'
if "bucket" not in df_prices.columns:
    df_prices = df_prices.copy()
    ts_col = "timestamp" if "timestamp" in df_prices.columns else "date"
    df_prices[ts_col] = pd.to_datetime(df_prices[ts_col], errors="coerce")
    df_prices["bucket"] = df_prices[ts_col].dt.to_period(PERIOD).dt.to_timestamp()

df_prices = df_prices.sort_values(["asset","bucket"])
df_prices["return"] = df_prices.groupby("asset")["price"].pct_change()
HORIZON = 1  # ex: 1 jour si PERIOD="D", 1 mois si PERIOD="M"
df_prices["target_return_next"] = df_prices.groupby("asset")["return"].shift(-HORIZON)
df_prices["target_up_next"] = (df_prices["target_return_next"] > 0).astype(int)

# 3) MERGE sur bucket uniquement (sentiment global répliqué sur tous les assets)
df_all = (
    df_prices.merge(feat_text, on="bucket", how="left")
             .dropna(subset=["target_return_next"])
)

# 4) vérif colonnes requises
num_cols = ["pos_mean","neu_mean","neg_mean","polarity_mean","n_docs"]
for c in ["text"] + num_cols:
    if c not in df_all.columns:
        raise KeyError(f"Colonne manquante après merge: {c}")

# 5) prêt pour featurisation
X_df = df_all[["text"] + num_cols].copy()
y_reg = df_all["target_return_next"].values
y_cls = df_all["target_up_next"].values

print(df_all.head())



    asset     bucket     price    return  target_return_next  target_up_next  \
0  EURUSD 2003-12-01  1.196501       NaN            0.010360               1   
1  EURUSD 2003-12-02  1.208897  0.010360            0.002813               1   
2  EURUSD 2003-12-03  1.212298  0.002813           -0.003467               0   
3  EURUSD 2003-12-04  1.208094 -0.003467            0.008775               1   
4  EURUSD 2003-12-05  1.218695  0.008775            0.002713               1   

   pos_mean  neu_mean  neg_mean  polarity_mean  n_docs  \
0   0.94682  0.030588  0.022592       0.924229     1.0   
1       NaN       NaN       NaN            NaN     NaN   
2       NaN       NaN       NaN            NaN     NaN   
3       NaN       NaN       NaN            NaN     NaN   
4       NaN       NaN       NaN            NaN     NaN   

                                                text  
0  Jean-Claude Trichet, President of the European...  
1                                                NaN  
2    

In [ ]:
# =========================
# TF-IDF (char_wb) + SVD  |  Transformer scikit-learn
# =========================
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.utils.validation import check_is_fitted

from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

# ==== 1) Transformers ====

class TfidfSvdFeaturizer(BaseEstimator, TransformerMixin):
    """
    Transforme X['text'] -> matrice dense (n_samples, n_components)
    via TF-IDF (char_wb n-grams) puis TruncatedSVD (LSA).
    Compatible avec Grid/RandomizedSearch (hyperparamètres exposés en __init__).
    """
    def __init__(self,
                 min_df=5,
                 max_df=0.99,
                 ngram_range=(3, 5),
                 n_components=100,
                 lowercase=True,
                 random_state=42):
        # hyperparamètres (doivent être stockés en attributs pour sklearn.clone)
        self.min_df = min_df
        self.max_df = max_df
        self.ngram_range = ngram_range
        self.n_components = n_components
        self.lowercase = lowercase
        self.random_state = random_state

        # objets appris (suffixe _)
        self.vectorizer_ = None
        self.svd_ = None

    def fit(self, X, y=None):
        # X est un DataFrame avec une colonne "text"
        text = X["text"].fillna("").astype(str).values

        self.vectorizer_ = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=self.ngram_range,
            min_df=self.min_df,
            max_df=self.max_df,
            lowercase=self.lowercase,
        )
        X_tfidf = self.vectorizer_.fit_transform(text)

        self.svd_ = TruncatedSVD(
            n_components=self.n_components,
            random_state=self.random_state
        )
        self.svd_.fit(X_tfidf)
        return self

    def transform(self, X):
        check_is_fitted(self, ["vectorizer_", "svd_"])
        text = X["text"].fillna("").astype(str).values
        X_tfidf = self.vectorizer_.transform(text)
        X_svd = self.svd_.transform(X_tfidf)  # ndarray dense
        return X_svd


class NumSelector(BaseEstimator, TransformerMixin):
    """
    Sélectionne des colonnes numériques dans X et renvoie un ndarray dense.
    """
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[self.cols].astype(float).fillna(0.0).values


# ==== 2) Exemple d’utilisation (classification ou régression) ====

# -- Prépare X / y (ex: à partir de df_all)
# df_all doit contenir la colonne 'text' + tes features numériques
# et les cibles target_up_next (classification) ou target_return_next (régression).
num_cols = ["pos_mean", "neu_mean", "neg_mean", "polarity_mean", "n_docs"]

# Exemple: tâche
TASK = "classification"   # ou "regression"

if TASK == "classification":
    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    scoring = "roc_auc"
    target_col = "target_up_next"
    param_space = {
        "clf__C": np.logspace(-2, 2, 10),
        "features__tfidfsvd__n_components": [50,100,200],
        "features__tfidfsvd__min_df": [2,5,10],
        "features__tfidfsvd__max_df": [0.9, 0.99],
        "features__tfidfsvd__ngram_range": [(3,5),(4,6)],
    }
else:
    model = Ridge(alpha=1.0)
    scoring = "neg_mean_squared_error"
    target_col = "target_return_next"
    param_space = {
        "clf__alpha": np.logspace(-4, 0, 10),
        "features__tfidfsvd__n_components": [50,100,200],
    }

# X_df contient la colonne 'text' + num_cols
X_df = df_all[["text"] + num_cols].copy()
y = df_all[target_col].copy()

# -- Union de features : TF-IDF+SVD (texte) + features numériques
features_union = FeatureUnion([
    ("tfidfsvd", TfidfSvdFeaturizer(
        min_df=5,
        max_df=0.99,
        ngram_range=(3, 5),
        n_components=100,
        lowercase=True,
        random_state=42
    )),
    ("num", NumSelector(num_cols))
])

# -- Pipeline complet
pipe = Pipeline([
    ("features", features_union),
    # StandardScaler: with_mean=False par sécurité si une étape renvoie du sparse
    ("scale", StandardScaler(with_mean=False)),
    ("clf", model),
])

# -- Validation temporelle + recherche d’hyperparamètres
cv = TimeSeriesSplit(n_splits=5)

search = RandomizedSearchCV(
    pipe,
    param_distributions=param_space,
    n_iter=8,
    cv=cv,
    scoring=scoring,
    random_state=42,
    verbose=1,
    n_jobs=-1
)

search.fit(X_df, y)
print("Best params:", search.best_params_)
print("Best CV score:", search.best_score_)

# (optionnel) entraînement final sur tout l’historique avec les meilleurs params
best_pipe = search.best_estimator_
best_pipe.fit(X_df, y)


Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best params: {'clf__C': np.float64(35.93813663804626)}
Best CV score: 0.5033518892312419


,steps,"[('features', ...), ('scale', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformer_list,"[('tfidfsvd', ...), ('num', ...)]"
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,min_df,5
,max_df,0.99


In [ ]:
# --- Évaluation holdout (dernier 20%) ---
split_idx = int(len(df_all)*0.8)
X_train, X_test = X_df.iloc[:split_idx], X_df.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

final_model = search.best_estimator_.fit(X_train, y_train)

if TASK=="classification":
    proba = final_model.predict_proba(X_test)[:,1]
    pred = (proba>=0.5).astype(int)
    print("Accuracy:", accuracy_score(y_test, pred))
    print("ROC AUC:", roc_auc_score(y_test, proba))
else:
    pred = final_model.predict(X_test)
    print("R^2:", r2_score(y_test, pred))
    print("RMSE:", mean_squared_error(y_test, pred, squared=False))

Accuracy: 0.4786440677966102
ROC AUC: 0.5066073936458437


In [ ]:
# --- Export des scores in-sample & dernier jour ---
try:
    import joblib
    joblib.dump(final_model, "model.joblib")
    print("Modèle sauvegardé dans model.joblib")
except Exception as e:
    print("Sauvegarde modèle ignorée:", e)

if TASK=="classification":
    scores_all = final_model.predict_proba(X_df)[:,1]
else:
    scores_all = final_model.predict(X_df)

out = df_all[["asset","bucket"]].copy()
out["indicator"] = scores_all
out.to_csv("indicator_in_sample.csv", index=False)
print("Écrit: indicator_in_sample.csv")

last_mask = out.groupby("asset")["bucket"].transform(lambda s: s==s.max()).astype(bool)
last_scores = out[last_mask].sort_values(["asset","bucket"])
last_scores.to_csv("indicator_latest.csv", index=False)
print("Écrit: indicator_latest.csv")

Modèle sauvegardé dans model.joblib
Écrit: indicator_in_sample.csv
Écrit: indicator_latest.csv


: 